In [ ]:
import os
import shutil

import matplotlib.pyplot as plt
%env NX_CUGRAPH_AUTOCONFIG=True
import networkx as nx
#import nx_cugraph as nxcg
#import cudf
#import cugraph
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F



import random

if os.path.exists('generated_graphs'):
    shutil.rmtree('generated_graphs')  # Supprime tout le dossier et son contenu
os.makedirs('generated_graphs', exist_ok=True)  # Recrée un répertoire vide





def generate_random_bipartite_graph(n):
    partition_size = n // 2
    remainder = n % 2

    G = nx.bipartite.random_graph(partition_size, partition_size + remainder, 0.5)
    
    adj_matrix = nx.to_numpy_array(G)
    if adj_matrix.shape != (n, n):
        adj_matrix = np.pad(adj_matrix, ((0, n - adj_matrix.shape[0]), (0, n - adj_matrix.shape[1])), mode='constant')
    
    return adj_matrix

def generate_random_planar_graph(n):
    side_length = int(np.ceil(np.sqrt(n)))
    
    G = nx.grid_2d_graph(side_length, side_length)
    
    # Convertir les nœuds de la grille en un seul index
    mapping = {node: i for i, node in enumerate(G.nodes())}
    G = nx.relabel_nodes(G, mapping)
    
    # Si le nombre de nœuds est supérieur à n, supprimer les nœuds supplémentaires
    if len(G.nodes()) > n:
        nodes_to_remove = list(G.nodes())[n:]
        G.remove_nodes_from(nodes_to_remove)
    
    # Ajouter des arêtes aléatoires tout en vérifiant que le graphe reste planaire
    possible_edges = [(u, v) for u in range(n) for v in range(u + 1, n) if not G.has_edge(u, v)]
    np.random.shuffle(possible_edges)
    
    for u, v in possible_edges:
        G.add_edge(u, v)
        if not nx.check_planarity(G)[0]:
            G.remove_edge(u, v)
    
    adj_matrix = nx.to_numpy_array(G)
    
    return adj_matrix

def generate_random_cyclic_graph(n):
    G = nx.Graph()  # Créer un graphe non orienté
    G.add_nodes_from(range(n))

    # Ajouter des arêtes pour créer un cycle
    edges = [(i, (i + 1) % n) for i in range(n)]  # Cycle de base
    G.add_edges_from(edges)

    # Ajouter des arêtes supplémentaires aléatoires pour augmenter la complexité
    while len(G.edges) < n + np.random.randint(1, n): 
        u, v = np.random.choice(n, 2, replace=False)
        G.add_edge(u, v)

    return nx.to_numpy_array(G)

def generate_random_cycle_graph(n):
    G = nx.Graph()
    G.add_nodes_from(range(n))
    
    # Mélanger les sommets de manière aléatoire
    nodes = list(G.nodes())
    np.random.shuffle(nodes)
    
    # Ajouter des arêtes pour former un cycle
    edges = [(nodes[i], nodes[(i + 1) % n]) for i in range(n)]
    G.add_edges_from(edges)
    
    adj_matrix = nx.to_numpy_array(G)
    
    return adj_matrix

def generate_random_binary_tree(n):
    G = nx.DiGraph()  # Graphe orienté
    G.add_node(0)  # Ajouter la racine
    for i in range(1, n):
        parent = np.random.randint(0, i)  # Choisir un parent aléatoire
        # S'assurer que le parent n'a pas déjà deux enfants
        while G.out_degree(parent) >= 2:
            parent = np.random.randint(0, i)
        G.add_node(i)  # Ajouter le nœud
        G.add_edge(parent, i)  # Créer une arête orientée du parent vers l'enfant
    return nx.to_numpy_array(G)

def generate_random_tree(n):
    G = nx.Graph()
    G.add_nodes_from(range(n))
    
    # Ajouter des arêtes pour la connexité
    for i in range(1, n):
        u = np.random.randint(0, i)
        G.add_edge(u, i)
    
    # Ajouter des arêtes supplémentaires aléatoires pour augmenter la complexité
    num_edges = np.random.randint(n - 1, n * (n - 1) // 2)  # Nombre d'arêtes entre n-1 et n*(n-1)/2 (graphe complet)
    while G.number_of_edges() < num_edges:
        u, v = np.random.choice(n, 2, replace=False)
        if not G.has_edge(u, v):
            G.add_edge(u, v)
    
    T = nx.minimum_spanning_tree(G, algorithm='boruvka')
    
    adj_matrix = nx.to_numpy_array(T)
    if adj_matrix.shape != (n, n):
        adj_matrix = np.pad(adj_matrix, ((0, n - adj_matrix.shape[0]), (0, n - adj_matrix.shape[1])), mode='constant')
    
    return adj_matrix


# Enregistrer les graphes dans le dataset
def save_graphs(graph_dataset, n_graphs, n_nodes):
    for i in range(n_graphs):
        adj_matrix = graph_dataset[i].reshape(n_nodes, n_nodes)
        G = nx.from_numpy_array(adj_matrix)
        
        # Créer un fichier d'image pour chaque graphe
        if i % 10_000 == 0:
            plt.figure(figsize=(8, 6))
            try:
                nx.draw_planar(G, with_labels=True, node_color="lightblue", font_weight="bold", node_size=700, edge_color="gray")
            except nx.NetworkXException:
                # Si le graphe n'est pas planaire, utiliser un autre layout
                nx.draw(G, with_labels=True, node_color="lightblue", font_weight="bold", node_size=700, edge_color="gray")
            
            plt.title(f"Graphe {i + 1}")
            plt.show()

        # Enregistrer l'image dans le dossier 'generated_graphs'
        #plt.savefig(f'generated_graphs/graph_{i + 1}.png')
        plt.close()  # Fermer la figure pour libérer de la mémoire



class DegreeLayer(nn.Module):
    def _init_(self, n_nodes):
        super(DegreeLayer, self)._init_()
        self.n_nodes = n_nodes
        self.sum_layer = nn.Linear(n_nodes, 1, bias=False)
        # Set weights to 1 and freeze them
        self.sum_layer.weight.data.fill_(1)
        self.sum_layer.weight.requires_grad = False
        
    def forward(self, adj_matrix):
        batch_size = adj_matrix.size(0)
        degrees = torch.zeros(batch_size, self.n_nodes).to(adj_matrix.device)
        for i in range(self.n_nodes):
            degrees[:, i] = self.sum_layer(adj_matrix[:, i, :]).squeeze()
        return degrees

class Generator(nn.Module):
    def __init__(self, n_nodes):
        super(Generator, self).__init__()
        self.n_nodes = n_nodes
        self.fc = nn.Sequential(
            nn.Linear(n_nodes, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, n_nodes * n_nodes)
        )

    def forward(self, z):
        output = self.fc(z)
        adj_matrix = output.view(-1, self.n_nodes, self.n_nodes)
        adj_matrix = (adj_matrix + adj_matrix.transpose(1, 2)) / 2
        adj_matrix = torch.sigmoid(adj_matrix)
        adj_matrix = adj_matrix * (1 - torch.eye(self.n_nodes)).to(adj_matrix.device)
        #adj_matrix = (adj_matrix > 0.5).float()
        return adj_matrix

class Discriminator(nn.Module):
    def __init__(self, n_nodes):
        super(Discriminator, self).__init__()
        self.n_nodes = n_nodes
        self.fc = nn.Sequential(
            nn.Linear(n_nodes * n_nodes, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, adj_matrix):
        x = adj_matrix.view(-1, self.n_nodes * self.n_nodes)
        return self.fc(x)

def binary_tree_loss(adj_matrix):
    batch_size, n_nodes, _ = adj_matrix.size()
    
    degrees = torch.sum(adj_matrix, dim=2)
    degree_penalty = torch.sum(torch.sum(F.relu(degrees - 3.0) ** 2, dim=1), dim=0)
    
    connectivity = torch.zeros(adj_matrix.size(0)).to(adj_matrix.device)
    for i in range(adj_matrix.size(0)):
        G = nx.from_numpy_array(adj_matrix[i].detach().cpu().numpy())
        num_components = nx.number_connected_components(G)
        
        connectivity[i] = num_components
    loss_connexe = torch.sum((connectivity-1)**2)
    
    n_moins_1_penalty = torch.sum(degrees, dim=1)
    n_moins_1_penalty = torch.sum(torch.sum(n_moins_1_penalty - (n_nodes-1))**2)
    
    return degree_penalty + loss_connexe + n_moins_1_penalty,n_moins_1_penalty
def simple_from_numpy_array(A):
    """Convert a 2D NumPy array to a NetworkX graph.

    Parameters
    ----------
    A : 2D numpy.ndarray
        An adjacency matrix representation of a graph.

    Returns
    -------
    G : NetworkX Graph
        A NetworkX graph generated from the adjacency matrix.
    """
    G = nx.Graph()
    n, m = A.shape
    if n != m:
        raise nx.NetworkXError("Adjacency matrix is not square.")
    
    G.add_nodes_from(range(n))
    edges = zip(*np.nonzero(A))
    G.add_edges_from((u, v, {'weight': A[u, v]}) for u, v in edges)
    
    return G








def degree_loss(adj_matrix):
    degrees = torch.sum(adj_matrix, dim=2)
    degree_penalty = torch.sum(torch.sum((degrees - 2.0) ** 2, dim=1), dim=0)
    
    # Initialize the penalty for disconnected graphs
    connectivity = torch.zeros(adj_matrix.size(0)).to(adj_matrix.device)
    
    # Calculate the number of connected components for each graph
    for i in range(adj_matrix.size(0)):
        G = nx.from_numpy_array(adj_matrix[i].detach().cpu().numpy())
        num_components = nx.number_connected_components(G)
        
        connectivity[i] = num_components
    
    loss_connexe = torch.sum((connectivity-1)**2)
    total_penalty = degree_penalty + loss_connexe
    
    return total_penalty#,torch.sum((connectivity-1)**2)

def tree_loss(adj_matrix):
    connectivity = torch.zeros(adj_matrix.size(0)).to(adj_matrix.device)
    
    matrice = (adj_matrix>0.5).float() + adj_matrix - adj_matrix.detach()
    for i in range(adj_matrix.size(0)):

        
        G = nx.from_numpy_array(matrice[i].detach().cpu().numpy())
        num_components = nx.number_connected_components(G)
        
        connectivity[i] = num_components
    
    loss_connexe = torch.sum((connectivity-1)**2)

    degrees = torch.sum(adj_matrix, dim=2)
    somme_deg=torch.sum(degrees,dim=1)
    somme_deg_loss=torch.sum(torch.sum((somme_deg-2*(n_nodes-1))**2,dim=0))

    return loss_connexe + somme_deg_loss, loss_connexe

def bipartite_loss(adj_matrix):#ne marche pas encore
    batch_size, n_nodes, _ = adj_matrix.size()
    total_loss = torch.zeros(batch_size).to(adj_matrix.device)
    for i in range(batch_size):
        # Utiliser le straight-through estimator
        matrice = (adj_matrix[i] > 0.5).float() + adj_matrix[i] - adj_matrix[i].detach()
        
        # Convertir la matrice d'adjacence en un graphe NetworkX
        G = nx.from_numpy_array(matrice.detach().cpu().numpy())
        try:
            # Trouver les cycles dans le graphe
            cycles = nx.find_cycle(G)
            
            # Calculer la perte basée sur les arêtes qui forment les cycles
            cycle_loss = 0
            for u, v in cycles:
                cycle_loss += adj_matrix[i, u, v]
            
            total_loss[i] = cycle_loss ** 2
        except nx.NetworkXNoCycle:
            # Si aucun cycle n'est trouvé, pénaliser le graphe
            total_loss[i] = 0.0
        degrees = torch.sum(matrice, dim=1)
        degree_penalty = torch.sum((degrees < 1).float())
        
        total_loss[i] += degree_penalty
    
    
    return torch.sum(total_loss), torch.sum(total_loss)
            
def maximal_clique_time_loss(adj_matrix):
    batch_size, n_nodes, _ = adj_matrix.size()
    total_loss = torch.zeros(batch_size).to(adj_matrix.device)
    time_loss = torch.zeros(batch_size).to(adj_matrix.device)
    for i in range(batch_size):
        # Utiliser le straight-through estimator
        matrice = (adj_matrix[i] > 0.5).float() + adj_matrix[i] - adj_matrix[i].detach()
        # Convertir la matrice d'adjacence en un graphe NetworkX
        G = nx.from_numpy_array(matrice.detach().cpu().numpy())
        
        start_time = time.time()
        # Trouver les cycles dans le graphe
        maximal_cliques = nx.find_cliques(G)
        end_time = time.time()
        # Calculer la perte basée sur les arêtes qui forment les cycles
        time_loss[i] = end_time - start_time
    time_loss = -torch.sum(time_loss)
    return time_loss, time_loss

def train(n_nodes=10, n_graphs=256, num_epochs=10_000, batch_size=32, fct=generate_random_cycle_graph):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(device)
    
    graph_dataset = [fct(n_nodes) for _ in range(n_graphs)]
    
    generator = Generator(n_nodes).to(device)
    discriminator = Discriminator(n_nodes).to(device)
    
    g_optimizer = optim.Adam(generator.parameters(), lr=0.0001)
    d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0001)
    if fct==generate_random_cycle_graph:
        aux_loss = degree_loss
    elif fct==generate_random_binary_tree:
        print("binary tree")
        aux_loss = binary_tree_loss
    elif fct==generate_random_tree:
        print("tree")
        aux_loss = tree_loss
    elif fct==generate_random_bipartite_graph:
        print("bipartite")
        aux_loss = bipartite_loss
    for epoch in range(num_epochs):
        start_timeg = time.time()
        for i in range(0, n_graphs, batch_size):
            #real_graphs = torch.tensor(graph_dataset[i:i+batch_size], dtype=torch.float32).to(device)
            start_timeg = time.time()
            batch_graphs_np = np.array(graph_dataset[i:i+batch_size])
            real_graphs = torch.tensor(batch_graphs_np, dtype=torch.float32).to(device)

            
            
            z = torch.randn(batch_size, n_nodes).to(device)
            
            fake_adj = generator(z)
            

            d_optimizer.zero_grad()
            d_real = discriminator(real_graphs)
            d_fake = discriminator(fake_adj.detach())
            d_loss = -torch.mean(torch.log(d_real) + torch.log(1 - d_fake))
            d_loss.backward()
            d_optimizer.step()
            
            g_optimizer.zero_grad()
            g_fake = discriminator(fake_adj)
            gen_time = time.time() - start_timeg  
            start_time = time.time()
            custom_loss, n_moins_1 = aux_loss(fake_adj)
            loss_time = time.time() - start_time
            
            g_loss = -torch.mean(torch.log(1-g_fake)) + custom_loss
            g_loss.backward()
            g_optimizer.step()
            
            
        if epoch % 1 == 0:
            print(f'Epoch [{epoch}/{num_epochs}], d_loss: {d_loss.item()}, g_loss: {g_loss.item()}, aux_loss: {custom_loss.item()}, g_without_aux_loss: {g_loss.item() - custom_loss.item()}, n_moins_1_penalty: {n_moins_1.item()}')
            print(f'Time for generator: {gen_time:.4f}s, Time for tree_loss: {loss_time:.4f}s')
            torch.save(generator.state_dict(), 'generator_cycle2.pth')
            torch.save(discriminator.state_dict(), 'discriminator_cycle.pth')
n_nodes = 15
train(n_nodes=n_nodes, n_graphs=1024, num_epochs=50, batch_size=32, fct=generate_random_bipartite_graph)

def visualize_generated_graphs(generator, n_samples=5, n_nodes=10):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, n_nodes)
        fake_adj = (generator(z)>0.5).float()
        
        for i in range(n_samples):
            adj_matrix = fake_adj[i].numpy()
            G = nx.from_numpy_array(adj_matrix)
            degrees = torch.sum(fake_adj[i], dim=1).numpy().round(2)
            
            plt.figure(figsize=(8, 6))
            nx.draw(G, with_labels=True, node_color='lightblue', 
                   node_size=500, font_size=16, font_weight='bold')
            plt.title(f'Generated Graph {i+1}\nNode Degrees: {degrees}')
            plt.show()

generator = Generator(n_nodes=n_nodes)

generator.load_state_dict(torch.load('generator_cycle2.pth'))

visualize_generated_graphs(generator, n_samples=5, n_nodes=n_nodes)